# Addendum to "Advanced Machine Learning Techniques"

## Starter code

In [1]:
import pandas as pd
import numpy as np

In [2]:
def make_features(df):
    df['num_ingredients'] = df.ingredients.apply(len)
    df['ingredient_length'] = df.ingredients.apply(lambda x: np.mean([len(item) for item in x]))
    df['ingredients_str'] = df.ingredients.astype('str')
    return df

In [3]:
train = make_features(pd.read_json('../data/train.json'))
y = train['cuisine']

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [5]:
vect = CountVectorizer(token_pattern=r"'([a-z ]+)'")
nb = MultinomialNB()

## Part 6 rewritten using `ColumnTransformer`

[make_column_transformer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.compose.make_column_transformer.html)

In [6]:
from sklearn.compose import make_column_transformer

In [7]:
# vectorize 1 column, passthrough 2 columns, and drop the remaining columns
ct = make_column_transformer(
    (vect, 'ingredients_str'),
    ('passthrough', ['num_ingredients', 'ingredient_length']),
    remainder='drop')

In [8]:
# create the feature matrix from the DataFrame
X_dtm_manual = ct.fit_transform(train)
X_dtm_manual.shape

(39774, 6252)

### Cross-validation

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

In [10]:
# create a pipeline of the ColumnTransformer and Naive Bayes
pipe = make_pipeline(ct, nb)

In [11]:
# properly cross-validate the entire pipeline
cross_val_score(pipe, train, y, cv=5, scoring='accuracy').mean()

0.7134318388611878

### Alternative way to specify `Pipeline` and `ColumnTransformer`

[Pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) and [ColumnTransformer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [13]:
# duplicate the pipeline structure without using make_pipeline or make_column_transformer
pipe = Pipeline([
    ('columntransformer', ColumnTransformer([
            ('countvectorizer', vect, 'ingredients_str'),
            ('passthrough', 'passthrough', ['num_ingredients', 'ingredient_length'])],
            remainder='drop')),
    ('multinomialnb', nb)
])

### Grid search of a nested `Pipeline`

In [14]:
# examine the pipeline steps
pipe.steps

[('columntransformer',
  ColumnTransformer(n_jobs=None, remainder='drop', sparse_threshold=0.3,
                    transformer_weights=None,
                    transformers=[('countvectorizer',
                                   CountVectorizer(analyzer='word', binary=False,
                                                   decode_error='strict',
                                                   dtype=<class 'numpy.int64'>,
                                                   encoding='utf-8',
                                                   input='content',
                                                   lowercase=True, max_df=1.0,
                                                   max_features=None, min_df=1,
                                                   ngram_range=(1, 1),
                                                   preprocessor=None,
                                                   stop_words=None,
                                                   strip_accent

In [15]:
# create a grid of parameters to search (and specify the pipeline step along with the parameter)
param_grid = {}
param_grid['columntransformer__countvectorizer__token_pattern'] = [r"\b\w\w+\b", r"'([a-z ]+)'"]
param_grid['multinomialnb__alpha'] = [0.5, 1]
param_grid

{'columntransformer__countvectorizer__token_pattern': ['\\b\\w\\w+\\b',
  "'([a-z ]+)'"],
 'multinomialnb__alpha': [0.5, 1]}

In [16]:
from sklearn.model_selection import GridSearchCV

In [17]:
# pass the pipeline to GridSearchCV
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(train, y);

In [18]:
# print the single best score and parameters that produced that score
print(grid.best_score_)
print(grid.best_params_)

0.7426710916679238
{'columntransformer__countvectorizer__token_pattern': "'([a-z ]+)'", 'multinomialnb__alpha': 0.5}
